# 📊 Integração Python + Excel

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Pandas](https://img.shields.io/badge/Pandas-150458?style=flat-square&logo=pandas&logoColor=white)
![Openpyxl](https://img.shields.io/badge/Openpyxl-217346?style=flat-square&logo=microsoft-excel&logoColor=white)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-excel%20%7C%20pandas%20%7C%20openpyxl-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-dataframe%20%7C%20read__csv-purple)
![Biblioteca](https://img.shields.io/badge/requer-pip%20install%20pandas%20&&_openpyxl-orange)

> Até aqui trabalhamos sempre com `.csv`. Mas o Excel (`.xlsx`) é um formato mais "vivo": tem fórmulas, formatação, várias abas. Existem duas formas de mexer nele em Python, e escolher a errada pode custar caro (literalmente apagar fórmulas de uma planilha).


## 📋 Conteúdo

1. [Introdução: pandas x openpyxl](#-1-introdução-pandas-x-openpyxl)
2. [Desafio](#-2-desafio)
3. [Solução com pandas](#-3-solução-com-pandas)
4. [Solução com openpyxl](#-4-solução-com-openpyxl)
5. [Pandas x openpyxl: o que aconteceu com a fórmula?](#-5-pandas-x-openpyxl-o-que-aconteceu-com-a-fórmula)

## 🐼📗 1. Introdução: pandas x openpyxl

Existem 2 formas principais de integrar Python com Excel:

| Método: | 🐼 Pandas | 📗 Openpyxl |
|---|---|---|
| Como enxerga o arquivo | Como uma **base de dados** (linhas/colunas) | Como uma **planilha** (células, abas, fórmulas) |
| Uso mais comum | Análise e manipulação de dados | Edição pontual, "como se fosse um VBA" |
| Fórmulas do Excel | Lê o **valor calculado**, não a fórmula | Preserva a fórmula original |
| Formatação (cor, negrito...) | Perde ao salvar de novo | Mantém, se você não mexer nela |
| Performance | Mais rápido para grandes volumes | Mais lento, célula por célula |
| Estrutura original do arquivo | Pode desmontar a estrutura ao salvar | Mantém mais a estrutura, mas **teste sempre** |

> 💡 **Dica:** não é "um ou outro" para sempre — a escolha depende do que você precisa fazer com aquele arquivo específico.

## 🎯 2. Desafio

Temos uma planilha (`Produtos.xlsx`) com produtos e serviços. Com o aumento do imposto sobre **serviços**, precisamos atualizar o preço dos itens impactados pela mudança.

**Novo Multiplicador Imposto: 1.5**

Regras:
- Só os itens com `Tipo == 'Serviço'` são afetados.
- `Produtos` (Tipo `Produto`) mantêm o multiplicador original.
- O `Preço Base Reais` precisa refletir o novo multiplicador.

Vamos resolver o mesmo desafio das duas formas, para comparar o resultado.

In [ ]:
import pandas as pd
from cores import *

produtos_df = pd.read_excel('../spec/Produtos.xlsx')

display(produtos_df)

## 🐼 3. Solução com pandas

Com o método: `.loc[]`, filtramos as linhas de `Serviço` e atualizamos as duas colunas de uma vez: o multiplicador e o preço em reais (recalculado a partir do preço original).

```python
dataframe.loc[condição, 'coluna'] = novo_valor
```

| Parâmetro | O que faz |
|-----------|-----------|
| `condição` | Filtro booleano — aqui, `Tipo == 'Serviço'` |
| `'coluna'` | Coluna(s) que serão atualizadas nas linhas filtradas |
| `novo_valor` | Valor (ou Series) atribuído às linhas filtradas |

In [ ]:
eh_servico = produtos_df['Tipo'] == 'Serviço'

produtos_df.loc[eh_servico, 'Multiplicador Imposto'] = 1.5
produtos_df.loc[eh_servico, 'Preço Base Reais'] = (
    produtos_df['Preço Base Original'] * produtos_df['Multiplicador Imposto']
)

display(produtos_df)

> ⚠️ **Cuidado:** repare que a coluna `Preço Base Reais` do arquivo original guarda uma **fórmula** (`=D2*B2`), não um número fixo. O `pd.read_excel` já traz o valor calculado — a fórmula em si se perde assim que os dados viram um DataFrame.

In [5]:
produtos_df.to_excel('../spec/Produtos - Atualizado (pandas).xlsx', index=True)

## 📗 4. Solução com openpyxl

Aqui tratamos o arquivo como uma planilha mesmo: navegamos célula a célula pela aba (`worksheet`) e só tocamos no que precisa mudar — a coluna `Multiplicador Imposto` (coluna `D`) das linhas de `Serviço`.

```python
workbook = openpyxl.load_workbook('arquivo.xlsx')
worksheet = workbook.active
worksheet.cell(row=linha, column=coluna).value = novo_valor
workbook.save('arquivo.xlsx')
```

| Parâmetro 🔑 | O que faz 🔓 |
|-----------|-----------|
| `load_workbook()` | Abre o arquivo `.xlsx` mantendo fórmulas e formatação |
| `workbook.active` | Aba ativa da planilha (aqui, a única) |
| `worksheet.cell(row, column)` | Acessa uma célula específica pela posição (1-indexado) |
| `.value` | Lê ou escreve o conteúdo da célula |

<br>

> 💡 **Dica:** como não tocamos na coluna `Preço Base Reais`, a fórmula `=D*B` continua lá — o próprio Excel recalcula o resultado sozinho na próxima vez que o arquivo for aberto.

In [ ]:
import openpyxl

workbook = openpyxl.load_workbook('../spec/Produtos.xlsx')
worksheet = workbook.active

coluna_tipo = 3 # C: Tipo
coluna_multiplicador = 4 # D: Multiplicador de Imposto

for linha in range(2, worksheet.max_row + 1):
    tipo = worksheet.cell(row=linha, column=coluna_tipo).value
    if (tipo == 'Serviço'):
        worksheet.cell(row=linha, column=coluna_multiplicador).value = 1.5

workbook.save('../spec/Produtos - Atualizado (openpyxl).xlsx')
print(f'{CinzaClaro}Planilha atualizada e salva com{Reset} {Verde}sucesso ✅{Reset}')

## 🔍 5. Pandas x openpyxl: o que aconteceu com a fórmula?

Vamos conferir a diferença na prática, olhando a célula `E3` (Pós Graduação) nos dois arquivos exportados.

In [ ]:
def valor_preco_base_reais(worksheet, produto):
    coluna_preco = next(
        celula.column_letter
        for celula in worksheet[1]
        if celula.value == 'Preço Base Reais'
    )
    coluna_produto = next(
        celula.column_letter
        for celula in worksheet[1]
        if celula.value == 'Produtos'
    )
    linha = next(
        celula.row
        for celula in worksheet[coluna_produto][1:]
        if celula.value == produto
    )
    return worksheet[f'{coluna_preco}{linha}'].value

wb_pandas = openpyxl.load_workbook('../spec/Produtos - Atualizado (pandas).xlsx')
wb_openpyxl = openpyxl.load_workbook('../spec/Produtos - Atualizado (openpyxl).xlsx')

valor_pandas = valor_preco_base_reais(wb_pandas.active, 'Pós Graduação')
valor_openpyxl = valor_preco_base_reais(wb_openpyxl.active, 'Pós Graduação')

print(f"{Vermelho}pandas   -> Preço Base Reais: {valor_pandas!r}{Reset}")
print(f"{Verde}openpyxl -> Preço Base Reais: {valor_openpyxl!r}{Reset}")

- No arquivo salvo pelo **pandas**, `E3` virou o número `6750.0` — a fórmula não existe mais.
- No arquivo salvo pelo **openpyxl**, `E3` continua sendo `=D3*B3` — a fórmula sobreviveu, e o Excel recalcula o valor sozinho ao abrir o arquivo.

> ⚠️ **Cuidado:** essa é a diferença que mais pega gente desprevenida. Se o arquivo tem fórmulas, formatação condicional ou macros que precisam ser preservadas, `openpyxl` é o caminho — o `pandas` é ótimo para análise, mas trata o Excel como dado puro.